In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
import glob
import tqdm

In [2]:


def plot(mean_dict, ci95_dict, length_dict, title, out_path):

    plt.rcParams.update({'font.size': 24})
    fig, ax = plt.subplots(figsize=(20,10))

    colour_list = ['royalblue', 'tomato', 'skyblue', 'lightsalmon', 'orange']

    for i, key in enumerate(mean_dict.keys()):
        colour = colour_list[i]
        mean = mean_dict[key]
        ci95 = ci95_dict[key]
        length = length_dict[key]
        width = len(mean)
        ax.plot(mean, color=colour, label=f"{key} (n={length:,})")
        ax.fill_between(np.arange(width), mean-ci95, mean+ci95,
                        color=colour, alpha=0.3)

    ax.set_title(title)
    ax.set_xlabel("Position")
    ax.set_ylabel(title)

    ax.set_xticks(np.arange(0, width, 3))
    ax.set_xticklabels(np.arange(0, width, 3))


    ## Vline at center
    ax.axvline(x=width//2, color="black", linestyle="--")

    ax.legend()
    plt.savefig(f"{out_path}/{title}_runs.pdf")
    plt.savefig(f"{out_path}/{title}_runs.png", dpi=300)
    plt.close()

    return None


In [3]:

feature_list = ["signal_mean","signal_median","signal_std","signal_len_log10","signal_amp","signal_rms","bq"]

ylim_dict = {}
ON0092_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0092/ON0092/result/dorado-070/intermediates/segmented_tokenized/signal_analysis/sampled.pkl")
ON0093_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0093/ON0093/result/dorado-070/intermediates/segmented_tokenized/signal_analysis/sampled.pkl")
ON0096_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0096/result/dorado-070/intermediates/segmented_tokenized/signal_analysis/sampled.pkl")
ON0098_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0098/result/dorado-070/intermediates/segmented_tokenized/signal_analysis/sampled.pkl")
ON0099_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0099/result/dorado-070/intermediates/segmented_tokenized/signal_analysis/sampled.pkl")
df_dict = {"ON0092": ON0092_df, "ON0093": ON0093_df, "ON0096": ON0096_df, "ON0098": ON0098_df, "ON0099": ON0099_df}

In [13]:
print(ON0092_df)

                                                       bq  \
0       [38, 38, 37, 37, 38, 38, 39, 43, 44, 39, 40, 3...   
1       [25, 27, 28, 29, 39, 44, 42, 47, 38, 39, 37, 3...   
2       [31, 30, 32, 32, 33, 34, 38, 38, 43, 39, 38, 3...   
3       [24, 34, 31, 30, 29, 29, 30, 31, 31, 30, 27, 2...   
4       [29, 34, 33, 31, 31, 32, 33, 40, 40, 38, 33, 3...   
...                                                   ...   
614395  [38, 38, 39, 40, 38, 39, 37, 38, 37, 38, 40, 3...   
614396  [34, 34, 32, 33, 33, 33, 34, 38, 39, 39, 38, 3...   
614397  [25, 25, 26, 30, 25, 24, 24, 18, 18, 18, 22, 1...   
614398  [29, 29, 28, 16, 16, 15, 14, 13, 13, 15, 17, 1...   
614399  [33, 33, 33, 32, 31, 31, 31, 30, 32, 33, 34, 3...   

                        motif  cb_idx  \
0       UCAGGCUGAAAAAAUUUAAUC       0   
1       UUUGAGAAAAAAACAUUAGGC       0   
2       CGACCGUCAAAAAUGCAAUGC       0   
3       GAGCGCAUAAAAAGGGGCGAC       0   
4       AAGCCAAGAAAAAGCGCUGUC       0   
...                  

In [5]:
out_path = "/extdata4/baeklab/Hyeonseo/m6A/plot/feature_analysis"
os.makedirs(out_path, exist_ok = True)
for feature in ["signal_mean","bq"]:
    feature_renamed = f"{feature}"

    data_dict = {df_name: np.stack(df[feature].values) for df_name, df in df_dict.items()}

    mean_dict = {df_name: np.mean(data, axis=0) for df_name, data in data_dict.items()}
    ci95_dict = {df_name: 1.96*np.std(data, axis=0)/np.sqrt(data.shape[0]) for df_name, data in data_dict.items()}
    length_dict = {df_name: data.shape[0] for df_name, data in data_dict.items()}

    plot(mean_dict, ci95_dict, length_dict, feature_renamed, out_path)


In [6]:
out_path = "/extdata4/baeklab/Hyeonseo/m6A/plot/feature_analysis"
os.makedirs(out_path, exist_ok = True)

a_df = pd.concat([ON0092_df, ON0096_df]).copy().reset_index(drop=True)
m6a_df = []
for df in [ON0093_df, ON0098_df, ON0099_df]:
    df = df.groupby("5mer")
    for _, sub_df in tqdm.tqdm(df, total=df.ngroups):
        m6a_df.append(sub_df.sample(1600))
m6a_df = pd.concat(m6a_df).copy().reset_index(drop=True)

df_dict_2 = {"A": a_df,
             "m6A": m6a_df}

print(a_df)
print(m6a_df)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:00<00:00, 594.97it/s]


                                                        bq  \
0        [38, 38, 37, 37, 38, 38, 39, 43, 44, 39, 40, 3...   
1        [25, 27, 28, 29, 39, 44, 42, 47, 38, 39, 37, 3...   
2        [31, 30, 32, 32, 33, 34, 38, 38, 43, 39, 38, 3...   
3        [24, 34, 31, 30, 29, 29, 30, 31, 31, 30, 27, 2...   
4        [29, 34, 33, 31, 31, 32, 33, 40, 40, 38, 33, 3...   
...                                                    ...   
1228795  [39, 34, 34, 33, 31, 30, 30, 31, 32, 34, 35, 3...   
1228796  [38, 38, 38, 37, 37, 37, 36, 33, 31, 16, 15, 1...   
1228797  [35, 35, 36, 37, 39, 42, 43, 41, 39, 36, 33, 3...   
1228798  [39, 38, 38, 39, 39, 39, 38, 38, 38, 38, 39, 3...   
1228799  [32, 32, 31, 31, 31, 31, 33, 35, 33, 32, 28, 2...   

                         motif  cb_idx  \
0        UCAGGCUGAAAAAAUUUAAUC       0   
1        UUUGAGAAAAAAACAUUAGGC       0   
2        CGACCGUCAAAAAUGCAAUGC       0   
3        GAGCGCAUAAAAAGGGGCGAC       0   
4        AAGCCAAGAAAAAGCGCUGUC       0   
...

In [7]:
for feature in ["bq"]:
    for motif in a_df["5mer"].unique():
        feature_renamed = f"{feature}_{motif}"
        df_dict_motif = {df_name: df[df["5mer"]==motif] for df_name, df in df_dict_2.items()}
        data_dict = {df_name: np.stack(df[feature].values) for df_name, df in df_dict_motif.items()}
        mean_dict = {df_name: np.mean(data, axis=0) for df_name, data in data_dict.items()}
        ci95_dict = {df_name: 1.96*np.std(data, axis=0)/np.sqrt(data.shape[0]) for df_name, data in data_dict.items()}
        length_dict = {df_name: data.shape[0] for df_name, data in data_dict.items()}
        plot(mean_dict, ci95_dict, length_dict, feature_renamed, out_path)



In [10]:
a_df["5mer_start"] = a_df["motif"].apply(lambda x: x[:5])
m6a_df["5mer_start"] = m6a_df["motif"].apply(lambda x: x[:5])

a_df["5mer_end"] = a_df["motif"].apply(lambda x: x[-5:])
m6a_df["5mer_end"] = m6a_df["motif"].apply(lambda x: x[-5:])

print(a_df)

                                                        bq  \
0        [38, 38, 37, 37, 38, 38, 39, 43, 44, 39, 40, 3...   
1        [25, 27, 28, 29, 39, 44, 42, 47, 38, 39, 37, 3...   
2        [31, 30, 32, 32, 33, 34, 38, 38, 43, 39, 38, 3...   
3        [24, 34, 31, 30, 29, 29, 30, 31, 31, 30, 27, 2...   
4        [29, 34, 33, 31, 31, 32, 33, 40, 40, 38, 33, 3...   
...                                                    ...   
1228795  [39, 34, 34, 33, 31, 30, 30, 31, 32, 34, 35, 3...   
1228796  [38, 38, 38, 37, 37, 37, 36, 33, 31, 16, 15, 1...   
1228797  [35, 35, 36, 37, 39, 42, 43, 41, 39, 36, 33, 3...   
1228798  [39, 38, 38, 39, 39, 39, 38, 38, 38, 38, 39, 3...   
1228799  [32, 32, 31, 31, 31, 31, 33, 35, 33, 32, 28, 2...   

                         motif  cb_idx  \
0        UCAGGCUGAAAAAAUUUAAUC       0   
1        UUUGAGAAAAAAACAUUAGGC       0   
2        CGACCGUCAAAAAUGCAAUGC       0   
3        GAGCGCAUAAAAAGGGGCGAC       0   
4        AAGCCAAGAAAAAGCGCUGUC       0   
...

In [11]:
a_df_aaaaa = a_df[a_df["5mer_start"]=="AAAAA"]
m6a_df_aaaaa = m6a_df[m6a_df["5mer_start"]=="AAAAA"]

print(len(a_df_aaaaa))
print(len(m6a_df_aaaaa))

1241
921


In [12]:
feature_renamed = f"{feature}_start_aaaaa"
df_dict_motif = {"A": a_df_aaaaa,
                    "m6A": m6a_df_aaaaa}
data_dict = {df_name: np.stack(df[feature].values) for df_name, df in df_dict_motif.items()}
mean_dict = {df_name: np.mean(data, axis=0) for df_name, data in data_dict.items()}
ci95_dict = {df_name: 1.96*np.std(data, axis=0)/np.sqrt(data.shape[0]) for df_name, data in data_dict.items()}
length_dict = {df_name: data.shape[0] for df_name, data in data_dict.items()}
plot(mean_dict, ci95_dict, length_dict, feature_renamed, out_path)

In [ ]:
block_ON0099_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0099/result/dorado-070/intermediates/block_df.pkl")
print(block_ON0099_df)
block_ON0098_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0098/result/dorado-070/intermediates/block_df.pkl")
print(block_ON0098_df)
block_ON0092_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0092/ON0092/result/dorado-070/intermediates/block_df.pkl")
print(block_ON0092_df)
block_ON0093_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0093/ON0093/result/dorado-070/intermediates/block_df.pkl")
print(block_ON0093_df)
block_ON0096_df = pd.read_pickle(f"/extdata4/baeklab/Hyeonseo/m6A/runs/exp_BB87/ON0096/result/dorado-070/dorado-070/intermediates/block_df.pkl")
print(block_ON0096_df)

          cb_idx  start_pos  end_pos  pos_RM  penalty     score  \
0              2        165      186     175        6  0.857143   
1              1        220      241     230        9  0.785714   
2              2        247      268     257        0  1.000000   
3              0        452      473     462        0  1.000000   
4              1        478      499     488        3  0.928571   
...          ...        ...      ...     ...      ...       ...   
77735403       2          9       30      19       10  0.761905   
77735404       2          5       26      15        9  0.785714   
77735405       0          9       30      19        4  0.904762   
77735406       1         14       35      24       10  0.761905   
77735407       1         12       33      22       15  0.642857   

                                       read_id  total_score  \
0         626a5a68-924d-4cde-96ad-1e1fa86ccf62    19.142857   
1         626a5a68-924d-4cde-96ad-1e1fa86ccf62    19.142857   
2     

In [14]:
print(a_df)
print(m6a_df)

                                                        bq  \
0        [38, 38, 37, 37, 38, 38, 39, 43, 44, 39, 40, 3...   
1        [25, 27, 28, 29, 39, 44, 42, 47, 38, 39, 37, 3...   
2        [31, 30, 32, 32, 33, 34, 38, 38, 43, 39, 38, 3...   
3        [24, 34, 31, 30, 29, 29, 30, 31, 31, 30, 27, 2...   
4        [29, 34, 33, 31, 31, 32, 33, 40, 40, 38, 33, 3...   
...                                                    ...   
1228795  [39, 34, 34, 33, 31, 30, 30, 31, 32, 34, 35, 3...   
1228796  [38, 38, 38, 37, 37, 37, 36, 33, 31, 16, 15, 1...   
1228797  [35, 35, 36, 37, 39, 42, 43, 41, 39, 36, 33, 3...   
1228798  [39, 38, 38, 39, 39, 39, 38, 38, 38, 38, 39, 3...   
1228799  [32, 32, 31, 31, 31, 31, 33, 35, 33, 32, 28, 2...   

                         motif  cb_idx  \
0        UCAGGCUGAAAAAAUUUAAUC       0   
1        UUUGAGAAAAAAACAUUAGGC       0   
2        CGACCGUCAAAAAUGCAAUGC       0   
3        GAGCGCAUAAAAAGGGGCGAC       0   
4        AAGCCAAGAAAAAGCGCUGUC       0   
...

In [15]:
def get_centre_motif(motif):
    convert_dict = {"A": "R", "C": "Y", "G": "R", "U": "Y"}
    centre_motif = f"{convert_dict[motif[1]]}A{convert_dict[motif[3]]}"
    return centre_motif

a_df["centre_motif"] = a_df["motif"].apply(get_centre_motif)
m6a_df["centre_motif"] = m6a_df["motif"].apply(get_centre_motif)

print(a_df)

                                                        bq  \
0        [38, 38, 37, 37, 38, 38, 39, 43, 44, 39, 40, 3...   
1        [25, 27, 28, 29, 39, 44, 42, 47, 38, 39, 37, 3...   
2        [31, 30, 32, 32, 33, 34, 38, 38, 43, 39, 38, 3...   
3        [24, 34, 31, 30, 29, 29, 30, 31, 31, 30, 27, 2...   
4        [29, 34, 33, 31, 31, 32, 33, 40, 40, 38, 33, 3...   
...                                                    ...   
1228795  [39, 34, 34, 33, 31, 30, 30, 31, 32, 34, 35, 3...   
1228796  [38, 38, 38, 37, 37, 37, 36, 33, 31, 16, 15, 1...   
1228797  [35, 35, 36, 37, 39, 42, 43, 41, 39, 36, 33, 3...   
1228798  [39, 38, 38, 39, 39, 39, 38, 38, 38, 38, 39, 3...   
1228799  [32, 32, 31, 31, 31, 31, 33, 35, 33, 32, 28, 2...   

                         motif  cb_idx  \
0        UCAGGCUGAAAAAAUUUAAUC       0   
1        UUUGAGAAAAAAACAUUAGGC       0   
2        CGACCGUCAAAAAUGCAAUGC       0   
3        GAGCGCAUAAAAAGGGGCGAC       0   
4        AAGCCAAGAAAAAGCGCUGUC       0   
...

In [16]:
for motif in ["RAR", "RAY", "YAR", "YAY"]:
    df_dict = {"A": a_df[a_df["centre_motif"]==motif],
                "m6A": m6a_df[m6a_df["centre_motif"]==motif]}
    for feature in ["bq", "signal_mean"]:
        feature_renamed = f"{feature}_{motif}"
        data_dict = {df_name: np.stack(df[feature].values) for df_name, df in df_dict.items()}
        mean_dict = {df_name: np.mean(data, axis=0) for df_name, data in data_dict.items()}
        ci95_dict = {df_name: 1.96*np.std(data, axis=0)/np.sqrt(data.shape[0]) for df_name, data in data_dict.items()}
        length_dict = {df_name: data.shape[0] for df_name, data in data_dict.items()}
        plot(mean_dict, ci95_dict, length_dict, feature_renamed, out_path)
    